In [ ]:
import json
from pathlib import Path

In [ ]:
DELHI_ALL_DATA = Path("/Users/hariomnarang/Desktop/personal/roads/mapillary_downloader/data/delhi/images")
CHUNKS_DEST = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi/chunks")

In [ ]:
from mtrain.utils import globL, mkdir
from itertools import batched
import shutil

images = globL(DELHI_ALL_DATA, "*.jpg")
len(images)

In [ ]:
from tqdm import tqdm
chunk_size = 500
batches = list(batched(images, chunk_size))
for i, batch in enumerate(tqdm(batches)):
    for image_path in batch:
        chunk_dest = mkdir(CHUNKS_DEST / str(i))
        dest = mkdir(chunk_dest / image_path.stem)
        shutil.copy(image_path, dest / "image.jpg")

In [ ]:
CHUNKS_DEST

In [ ]:
for i in range(66):
    ! (cd /Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi/chunks && dvctar add {i})

# run on chunks

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import show, DiskImage, DiskBooleanMask, overlay_mask_on_img as OV
from mtrain.example_dir import ExampleDir, load_npz
from mtrain.example_dir.iterdir import get_dirs
from mtrain.example_dir.defaults import default_negmask_learners, default_smallnet_learners
from tqdm import tqdm

In [ ]:
dirs = list(get_dirs(CHUNKS_DEST / "5"))

In [ ]:
negmask = default_negmask_learners(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models"), ["md", "high-recall", "unblurred"], 4)
smallnet = default_smallnet_learners(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models"), ["md", "sm"], 4)

In [ ]:
edirs = [ExampleDir(d, smallnet, negmask) for d in dirs]

In [ ]:
from tqdm import tqdm
for edir in tqdm(edirs):
    edir.smallnet_mask_path("md")
    edir.smallnet_mask_path("sm")
    edir.negmask_paths("md", "md")
    edir.negmask_paths("high-recall", "md")
    edir.negmask_paths("unblurred", "sm")